# Notes

code for producing GGIT euro gas tracker hydrogen pipelines summary stats, and for calculating landing page stats

In [1]:
import pandas
import numpy
import pygsheets
import datetime
import re
import pytz

In [2]:
# define the excel file to save tables in
current_time = datetime.datetime.now(pytz.timezone('US/Eastern')).strftime("%Y-%m-%d_T%H%M%S")

In [3]:
#fuel_type = 'Gas'
fuel_type = 'Hydrogen'
#fuel_type = 'Oil'
#fuel_type = 'NGL'

## import data

In [4]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA') # file to use for gas pipelines Dec 2023
spreadsheet = gc.open_by_key('1CktxlI1RgYUvKtL0iaNjnZszXFVjSj0JKgPnxkbm414') # Jan 2025 sheet
# spreadsheet = gc.open_by_key('1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs') # dec 2024 dataset for GGIT release

gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
oil_pipes = oil_pipes.drop('WKTFormat', axis=1)
pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#get other relevant sheets
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
owners_df_orig = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')

country_ratios_df = country_ratios_df.loc[country_ratios_df.Wiki!='']

# remove empty cells for pipes, owners
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
# pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Wiki']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel==fuel_type]

owners_df_orig = owners_df_orig.loc[owners_df_orig['ProjectID']!='']
# owners_df_orig = owners_df_orig.loc[owners_df_orig['Wiki']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig.Status!='N/A']

owners_df_orig.set_index('ProjectID', inplace=True)

parent_metadata_df = spreadsheet.worksheet('title', 'Parent metadata (3/3)').get_as_df(start='A3')
parent_metadata_df.set_index('Parent', inplace=True)

In [5]:
country_ratios_df.replace('--', numpy.nan, inplace=True)

owners_df_orig.replace('',numpy.nan,inplace=True)
owners_df_orig.replace('--',numpy.nan,inplace=True)

pipes_df_orig.replace('--',numpy.nan,inplace=True)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/1702877721.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df.replace('--', numpy.nan, inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/1702877721.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  owners_df_orig.replace('--',numpy.nan,inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/1702877721.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a f

In [6]:
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

#region_name = 'Global'; region_df_touse = region_df_orig.copy()
#region_name = 'AsiaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AsiaGasTracker=='Yes']
region_name = 'EuroGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.EuroGasTracker=='Yes']
#region_name = 'AfricaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AfricaGasTracker=='Yes']
# region_name = 'LatinAmericaTracker'; region_df_touse = region_df_orig.loc[region_df_orig.LatinAmericaTracker=='Yes']
#region_df_agt.copy()

#region_df_touse = region_df_orig.copy()

In [7]:
region_df_touse_cleaned = region_df_touse.loc[(region_df_touse.Region!='--')&
                                            (region_df_touse.SubRegion!='--')]
multiindex_region_subregion = region_df_touse_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion

MultiIndex([(  'Asia',    'Western Asia'),
            ('Europe',  'Eastern Europe'),
            ('Europe', 'Northern Europe'),
            ('Europe', 'Southern Europe'),
            ('Europe',  'Western Europe')],
           names=['Region', 'SubRegion'])

## file names with regional specifics

In [8]:
if fuel_type=='Gas':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-gas-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='Hydrogen':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-hydrogen-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='NGL':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-NGL-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='Oil':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-Oil-pipelines-'+str(datetime.date.today())+'.xlsx')

### create country-specific dataframes for region, country_ratios_df, owners_df

In [9]:
egt_projectids = pipes_df_orig.loc[pipes_df_orig.EuropeTracker=='yes'].ProjectID.tolist()
pci6_projectids = pipes_df_orig.loc[pipes_df_orig.PCI6=='yes'].ProjectID.tolist()
#country_ratios_df_touse = 

In [10]:
country_ratios_df_touse = country_ratios_df.loc[country_ratios_df.ProjectID.isin(egt_projectids)]
pipes_df_touse = pipes_df_orig.loc[pipes_df_orig.ProjectID.isin(egt_projectids)]

In [11]:
pipes_df_touse.loc[(pipes_df_touse.Fuel=='Hydrogen') & 
                    (pipes_df_touse.RouteType!='Included in other ProjectID') &
                    (pipes_df_touse.Status.isin(['proposed','construction']))].LengthMergedKm.sum()

np.float64(54401.0)

### sum LengthMergedKmByCountry and MergedKmByRegion

In [12]:
status_list = ['proposed', 
               'construction', 
               'shelved', 
               'cancelled', 
               'operating', 
               'idle', 
               'mothballed', 
               'retired']
country_list = sorted(list(set(country_ratios_df_touse['Country'])))
region_list = sorted(list(set(country_ratios_df_touse['Region'])))

In [13]:
excel_status_list = ['proposed', 
                     'construction', 
                     'in development (proposed + construction)', 
                     'shelved', 
                     'cancelled', 
                     'operating', 
                     'idle', 
                     'mothballed', 
                     'retired']
excel_status_list_with_countries = ['Country']+excel_status_list

In [14]:
# for hydrogen, avoid double counting
double_count_projectids = pipes_df_touse.loc[pipes_df_touse.RouteType=='Included in other ProjectID'].ProjectID.tolist()

In [15]:
country_ratios_df_subset = country_ratios_df_touse.loc[country_ratios_df_touse['Fuel']==fuel_type]
# SUBSET TO AVOID DOUBLE COUNTING:
country_ratios_df_subset = country_ratios_df_subset.loc[~country_ratios_df_subset.ProjectID.isin(double_count_projectids)]

km_by_country = pandas.DataFrame(columns=status_list, index=country_list)
km_by_region = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_country[status] = country_ratios_df_subset_status.groupby('Country')['LengthMergedKmByCountry'].sum()

print('===regional calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_region[status] = country_ratios_df_subset_status.groupby(['Region','SubRegion'])['LengthMergedKmByCountry'].sum()

# fille NaN with 0.0
km_by_region = km_by_region.fillna(0)
km_by_country = km_by_country.fillna(0)

km_by_region['in development (proposed + construction)'] = km_by_region[['proposed','construction']].sum(axis=1)
km_by_country['in development (proposed + construction)'] = km_by_country[['proposed','construction']].sum(axis=1)

km_by_country = km_by_country[excel_status_list]
km_by_region = km_by_region[excel_status_list]

km_by_region.index.names = ['Region','Subregion']
km_by_country.index.name = 'Country'

km_by_region.loc['Total',:] = km_by_region.sum(axis=0).values
km_by_country.loc['Total',:] = km_by_country.sum(axis=0).values

# drop all-zero rows
km_by_country = km_by_country.loc[~(km_by_country==0).all(axis=1)]

km_by_country.replace(0,'',inplace=True)
km_by_region.replace(0,'',inplace=True)

km_by_region.to_excel(excel_writer, 'Kilometers by region')
km_by_country.to_excel(excel_writer, 'Kilometers by country')

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired
===regional calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/266300265.py:42: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_region.to_excel(excel_writer, 'Kilometers by region')
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/266300265.py:43: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_country.to_excel(excel_writer, 'Kilometers by country')


In [16]:
# proposed infra
# cols: Country, PCI6, non-PCI, country total (km)

km_by_country_proposed_h2 = pandas.DataFrame(index=country_list)

country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset.Status.isin(['proposed'])]
km_by_country_proposed_h2['Proposed PCI6 km'] = country_ratios_df_subset_status.loc[country_ratios_df_subset_status.ProjectID.isin(pci6_projectids)].groupby('Country')['LengthMergedKmByCountry'].sum()
km_by_country_proposed_h2['Additional proposed km'] = country_ratios_df_subset_status.loc[~country_ratios_df_subset_status.ProjectID.isin(pci6_projectids)].groupby('Country')['LengthMergedKmByCountry'].sum()
km_by_country_proposed_h2['Total proposed km'] = km_by_country_proposed_h2.sum(axis=1)
km_by_country_proposed_h2.replace(numpy.nan, 0, inplace=True)
km_by_country_proposed_h2 = km_by_country_proposed_h2.sort_values('Total proposed km', ascending=False)
km_by_country_proposed_h2.to_excel('km_by_country_h2.xlsx')#, index=False)
km_by_country_proposed_h2.sum(axis=0)

Proposed PCI6 km          18462.33
Additional proposed km    31691.49
Total proposed km         50153.82
dtype: float64

In [17]:
km_by_country

,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired
Country,,,,,,,,,
Albania,175.98,,175.98,,,,,,
Austria,748.93,,748.93,,,,,,
Belgium,978.05,,978.05,,,,,,
Bosnia and Herzegovina,408.39,,408.39,,,,,,
Bulgaria,4475.54,,4475.54,,,,,,
Croatia,1842.14,,1842.14,,,,,,
Czech Republic,1417.88,,1417.88,,,,,,
Denmark,638.08,,638.08,50.06,,,,,
Estonia,267.1,,267.1,,,,,,


In [18]:
km_by_region

proposed construction  \
Region Subregion                                
Asia   Western Asia         0.63                
Europe Eastern Europe   11076.71                
       Northern Europe   6909.44                
       Southern Europe  15911.44                
       Western Europe   16236.80         30.0   
Total                   50135.02         30.0   

                        in development (proposed + construction)  shelved  \
Region Subregion                                                            
Asia   Western Asia                                         0.63            
Europe Eastern Europe                                   11076.71            
       Northern Europe                                   6909.44   967.16   
       Southern Europe                                  15911.44            
       Western Europe                                   16266.80   340.06   
Total                                                   50165.02  1307.22   

                       cancelled operating idle mothballed retired  
Region Subregion                                                    
Asia   Western Asia                                                 
Europe Eastern Europe                                               
       Northern Europe                                              
       Southern Europe                                              
       Western Europe                                               
Total

## pipeline km by parent company (owner) and project status

### first check that there are no missing projectids

In [19]:
owner_parent_calculations_df = pandas.DataFrame()
# needs country, km in each country columns as well

for idx,row in country_ratios_df_subset.iterrows():
    #print(row.ProjectID)
    parent_string = row.Parent
    #print(parent_string)

    #print(parent_string)
    # if the first letter of the parent_string is a chinese character, and it ends with [100.00%],
    # that means it's a non-researched QCC owner; keep as-is
    if re.findall(r'[\u4e00-\u9fff]+', parent_string[:1]) != [] and parent_string[-9:]=='[100.00%]':
        parent_list = [parent_string.split(' [100.00%]')[0]]
        percent_list = [1.0]
    # otherwise do the rest of the thing
    else:
        parent_list = re.sub(' \[.*?\]', '', parent_string).split('; ') # all entries must have a Owner [%] syntax
        percent_list = [float(i.rstrip('%'))/100. for i in re.findall('\\d+(?:\\.\\d+)?%', parent_string)]

    if parent_list.__len__()!=percent_list.__len__():
        #print(parent_list)
        if percent_list==[]:
            percent_list = [1/parent_list.__len__() for i in parent_list]
        else:
            nmissing = parent_list.__len__()-percent_list.__len__()
            # distribute nans evenly
            total = numpy.nansum(percent_list)
            leftover = 1-total
            percent_list += [leftover/nmissing]*nmissing
    for p_idx,parent in enumerate(parent_list):
        owner_parent_calculations_df = pandas.concat([owner_parent_calculations_df, 
                                                      pandas.DataFrame([{'Parent':parent, 'ProjectID':row.ProjectID, 
                                                                         'FractionOwnership':percent_list[p_idx],
                                                                         # 'ParentHQCountry':parent_metadata_df.loc[parent,'ParentHQCountry'],
                                                                         # 'ParentHQRegion':parent_metadata_df.loc[parent,'ParentHQRegion'],
                                                                         'Country':row.Country,
                                                                         'Status':row.Status,
                                                                         'LengthMergedKmByCountry':row.LengthMergedKmByCountry}])])

owner_parent_calculations_df['KmOwnership'] = owner_parent_calculations_df.FractionOwnership*owner_parent_calculations_df.LengthMergedKmByCountry

In [20]:
owner_parent_calculations_df

,Parent,ProjectID,FractionOwnership,Country,Status,LengthMergedKmByCountry,KmOwnership
0,Albgaz ShA,P0702,0.25,Croatia,proposed,262.01,65.5025
0,BH-Gas doo,P0702,0.25,Croatia,proposed,262.01,65.5025
0,Montenegro Bonus doo,P0702,0.25,Croatia,proposed,262.01,65.5025
0,Plinacro doo,P0702,0.25,Croatia,proposed,262.01,65.5025
0,Albgaz ShA,P0702,0.25,Montenegro,proposed,102.01,25.5025
...,...,...,...,...,...,...,...
0,Plinhold doo,P7091,1.00,Slovenia,proposed,25.00,25.0000
0,Plinhold doo,P7091,1.00,Hungary,proposed,25.00,25.0000
0,VNG AG,P7098,1.00,Germany,proposed,55.00,55.0000
0,unknown,P7103,1.00,Poland,proposed,425.95,425.9500


In [21]:
excel_status_list_with_countries

['Country',
 'proposed',
 'construction',
 'in development (proposed + construction)',
 'shelved',
 'cancelled',
 'operating',
 'idle',
 'mothballed',
 'retired']

In [22]:
#unique_owner_list = owner_parent_calculations_df.Parent.sort_values().unique().tolist()

##################################################
# create km count by owner, status
##################################################
owners_km_by_status_df = \
    owner_parent_calculations_df.groupby(
    ['Parent',
     # 'ParentHQCountry',
     'Status'])[['KmOwnership']].sum().unstack().droplevel(axis=1, level=[0])

owners_km_by_status_df = owners_km_by_status_df.reindex(columns=status_list)
owners_km_by_status_df = owners_km_by_status_df.reset_index().set_index('Parent')
owners_km_by_status_df.columns.name = None

owners_km_by_status_df['in development (proposed + construction)'] = owners_km_by_status_df[['proposed','construction']].sum(axis=1)

owners_km_by_status_df = owners_km_by_status_df.rename(columns={'Parent':'Parent Company',
                                                                          # 'ParentHQCountry':'Country'
                                                               })
# rearrange the order of the columns for output
owners_km_by_status_df = owners_km_by_status_df[excel_status_list]#_with_countries]

# totals_row = owners_ntrains_by_status_df.sum(axis=0, min_count=0)
# totals_row.name = 'Total'
# owners_ntrains_by_status_df = owners_ntrains_by_status_df.append(totals_row)
owners_km_by_status_df.loc['Total',:] = owners_km_by_status_df.sum(axis=0, min_count=0).values
owners_km_by_status_df.loc['Total','Country'] = ''

owners_km_by_status_df = owners_km_by_status_df.replace(numpy.nan, '')
owners_km_by_status_df = owners_km_by_status_df.replace(0, '')

owners_km_by_status_df.to_excel(excel_writer, sheet_name='Kilometers by owner')

### pipeline km by start year, type

In [23]:
pipes_started = pipes_df_touse.copy()
#pipes_started['StartYearLatest'].replace(numpy.nan,'',inplace=True)

if fuel_type == 'Gas':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Gas')]
if fuel_type == 'Oil':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Oil')]
if fuel_type == 'NGL':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='NGL')]

pipes_started_sum = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()

In [24]:
pipes_started_sum

StartYearEarliest
2024.0      352.00
2025.0     4393.45
2026.0     1514.49
2027.0     8672.04
2028.0     4309.46
2029.0    21191.34
2030.0    12963.16
2031.0     2541.82
2032.0     2978.81
2034.0      433.00
2035.0      954.93
2039.0     3648.20
2040.0     6422.93
2041.0      269.00
2043.0       94.00
2045.0      410.11
Name: LengthMergedKm, dtype: float64

In [25]:
if fuel_type == 'Gas':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Gas pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Gas pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'Hydrogen':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Hydrogen pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Hydrogen pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'Oil':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Oil pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Oil pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'NGL':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['NGL pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['NGL pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

km_by_start_year.loc['Total',:] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')
#km_by_start_year

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55123/1845660343.py:27: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')


In [26]:
km_by_start_year

,Hydrogen pipeline km
Start year,
1980,0.0
1981,0.0
1982,0.0
1983,0.0
1984,0.0
1985,0.0
1986,0.0
1987,0.0
1988,0.0


## save excel file

In [27]:
excel_writer.close()

## calculating stats for landing page

In [28]:
# number of projects tracked in total
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type].shape[0], 'hydrogen pipeline projects tracked')
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type]['LengthMergedKm'].sum()/1e6, 'M km tracked')

348 hydrogen pipeline projects tracked
0.07236423000000002 M km tracked


In [29]:
# number of projects tracked in total
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))].shape[0], 'hydrogen pipeline projects tracked')
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))]['LengthMergedKm'].sum()/1e3, 'K km tracked')

338 hydrogen pipeline projects tracked
69.679 K km tracked
